1. Import Libraries

In [1]:
import sqlite3
import csv
import json
from datetime import datetime

2. BankAccount Class (OOP Layer)

In [2]:
class BankAccount:
    def __init__(self, customer_name, account_type, balance, created_date):
        self.customer_name = customer_name
        self.account_type = account_type
        self.balance = balance
        self.created_date = created_date

    def __str__(self):
        return f"{self.customer_name} | {self.account_type} | {self.balance} | {self.created_date}"

    def to_dict(self):
        return {
            "customer_name": self.customer_name,
            "account_type": self.account_type,
            "balance": self.balance,
            "created_date": self.created_date
        }

    @staticmethod
    def from_dict(data):
        return BankAccount(
            data["customer_name"],
            data["account_type"],
            data["balance"],
            data["created_date"]
        )

In [23]:
b1=BankAccount("heer","Saving",1000,"2002-09-09")
print(b1)

heer | Saving | 1000 | 2002-09-09


3. Database Setup

In [3]:
def connect_db():
    return sqlite3.connect("bank.db")

def create_table():
    conn = connect_db()
    cur = conn.cursor()
    cur.execute("""
    CREATE TABLE IF NOT EXISTS accounts(
        account_id INTEGER PRIMARY KEY AUTOINCREMENT,
        customer_name TEXT,
        account_type TEXT,
        balance REAL,
        created_date TEXT
    )
    """)
    conn.commit()
    conn.close()

4. Validation Function

In [4]:
def validate_account_type(acc_type):
    return acc_type in ["Savings", "Current"]

def validate_amount(amount):
    return amount > 0

def validate_date(date):
    try:
        datetime.strptime(date, "%Y-%m-%d")
        return True
    except:
        return False

5. CRUD Operations

In [5]:
def create_account():
    name = input("Enter name: ")
    acc_type = input("Enter type (Savings/Current): ")
    balance = float(input("Enter balance: "))
    date = input("Enter date (YYYY-MM-DD): ")

    if not name or not validate_account_type(acc_type) or balance < 0 or not validate_date(date):
        print("Invalid input!")
        return

    conn = connect_db()
    cur = conn.cursor()
    cur.execute("INSERT INTO accounts (customer_name, account_type, balance, created_date) VALUES (?, ?, ?, ?)",
                (name, acc_type, balance, date))
    conn.commit()
    conn.close()
    print("Account created successfully!")

View Accounts

In [6]:
def view_accounts():
    conn = connect_db()
    cur = conn.cursor()
    cur.execute("SELECT * FROM accounts")
    rows = cur.fetchall()

    for row in rows:
        print(row)

    conn.close()

Update Account

In [7]:
def update_account():
    acc_id = int(input("Enter account ID: "))
    name = input("New name: ")
    acc_type = input("New type: ")

    conn = connect_db()
    cur = conn.cursor()
    cur.execute("UPDATE accounts SET customer_name=?, account_type=? WHERE account_id=?",
                (name, acc_type, acc_id))
    conn.commit()
    conn.close()

Delete Account

In [8]:
def delete_account():
    acc_id = int(input("Enter account ID: "))
    conn = connect_db()
    cur = conn.cursor()
    cur.execute("DELETE FROM accounts WHERE account_id=?", (acc_id,))
    conn.commit()
    conn.close()

6. Transactions

In [9]:
def deposit():
    acc_id = int(input("Enter account ID: "))
    amount = float(input("Enter amount: "))

    if not validate_amount(amount):
        print("Invalid amount")
        return

    conn = connect_db()
    cur = conn.cursor()

    cur.execute("SELECT balance FROM accounts WHERE account_id=?", (acc_id,))
    result = cur.fetchone()

    if result:
        new_balance = result[0] + amount
        cur.execute("UPDATE accounts SET balance=? WHERE account_id=?", (new_balance, acc_id))
        conn.commit()
        print("Deposit successful!")
    else:
        print("Account not found")

    conn.close()

Withdraw

In [10]:
def withdraw():
    acc_id = int(input("Enter account ID: "))
    amount = float(input("Enter amount: "))

    conn = connect_db()
    cur = conn.cursor()

    cur.execute("SELECT balance FROM accounts WHERE account_id=?", (acc_id,))
    result = cur.fetchone()

    if result:
        if amount > result[0]:
            print("Insufficient balance!")
        else:
            new_balance = result[0] - amount
            cur.execute("UPDATE accounts SET balance=? WHERE account_id=?", (new_balance, acc_id))
            conn.commit()
            print("Withdraw successful!")
    else:
        print("Account not found")

    conn.close()

7. Search

In [11]:
def search_account():
    choice = input("Search by 1.ID 2.Name: ")

    conn = connect_db()
    cur = conn.cursor()

    if choice == "1":
        acc_id = int(input("Enter ID: "))
        cur.execute("SELECT * FROM accounts WHERE account_id=?", (acc_id,))
    else:
        name = input("Enter name: ")
        cur.execute("SELECT * FROM accounts WHERE customer_name LIKE ?", ('%' + name + '%',))

    rows = cur.fetchall()
    for row in rows:
        print(row)

    conn.close()

🔹 8. Export Data

In [12]:
def export_csv():
    conn = connect_db()
    cur = conn.cursor()
    cur.execute("SELECT * FROM accounts")
    rows = cur.fetchall()

    with open("accounts.csv", "w", newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["id", "name", "type", "balance", "date"])
        writer.writerows(rows)

    conn.close()

JSON

In [13]:
def export_json():
    conn = connect_db()
    cur = conn.cursor()
    cur.execute("SELECT customer_name, account_type, balance, created_date FROM accounts")

    data = [dict(row) for row in cur.fetchall()]

    with open("accounts.json", "w") as f:
        json.dump(data, f)

    conn.close()

9. Menu (IMPORTANT FOR MARKS)

In [14]:
def menu():
    create_table()

    while True:
        print("\n1.Create 2.View 3.Search 4.Update 5.Delete")
        print("6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit")

        choice = input("Enter choice: ")

        try:
            if choice == "1":
                create_account()
            elif choice == "2":
                view_accounts()
            elif choice == "3":
                search_account()
            elif choice == "4":
                update_account()
            elif choice == "5":
                delete_account()
            elif choice == "6":
                deposit()
            elif choice == "7":
                withdraw()
            elif choice == "8":
                export_csv()
            elif choice == "9":
                export_json()
            elif choice == "10":
                break
            else:
                print("Invalid choice")
        except Exception as e:
            print("Error:", e)

10. Run Program

In [15]:
menu()


1.Create 2.View 3.Search 4.Update 5.Delete
6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit


Enter choice:  2



1.Create 2.View 3.Search 4.Update 5.Delete
6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit


Enter choice:  


Invalid choice

1.Create 2.View 3.Search 4.Update 5.Delete
6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit


...

Enter choice:  8



1.Create 2.View 3.Search 4.Update 5.Delete
6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit


Enter choice:  9


Error: dictionary update sequence element #0 has length 5; 2 is required

1.Create 2.View 3.Search 4.Update 5.Delete
6.Deposit 7.Withdraw 8.Export CSV 9.Export JSON 10.Exit


Enter choice:  10
